# Tørrfall vs. predicted substrate

Bruke tørrfallsone som hardbunn: how much of the intertidal `Tørrfall`
polygons (Kartverket sjøkart) is predicted as soft / mixture / hard by the
substrate model? Reported nationally and per marine water-type region.

Vector overlay is used throughout; the prediction layer is bbox-clipped to
the tørrfall extent (`.cx`) before overlay to avoid intersecting the whole
national polygon set.

Prediction file: `gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat_norge_2026_25833.geo.parquet`.
`LM_DK` is aggregated to soft / mixture / hard via the same grouping used
in `docs/substrat_model.md` (`DK_AB, DK_C, DK_D` → soft, `DK_0` → mixture,
`DK_EFGY` → hard).

In [1]:
import geopandas as gpd
import pandas as pd

import mnk.sources
from mnk.substrat.labelling import LM_DK_MAP

In [2]:
PRED_URL = 'gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat_norge_2026_25833.geo.parquet'

# Invert mnk.substrat.labelling.LM_DK_MAP (BunnType → LM_DK) and rename to English.
BUNN_TO_CLASS = {'løsbunn': 'soft', 'blanding': 'mixture', 'fastbunn': 'hard'}
LM_DK_TO_CLASS = {v: BUNN_TO_CLASS[k] for k, v in LM_DK_MAP.items()}
CLASS_ORDER = ['soft', 'mixture', 'hard']
print(LM_DK_TO_CLASS)

{'DK_AB, DK_C, DK_D': 'soft', 'DK_0': 'mixture', 'DK_EFGY': 'hard'}


In [3]:
%%time
torr = mnk.sources.sea_map_torrfall().to_crs('EPSG:25833')
torr = torr[torr.geometry.is_valid & ~torr.geometry.is_empty][['geometry']].copy()
torr['torr_area_m2'] = torr.geometry.area
print(f'Tørrfall polygons: {len(torr):,}, total area {torr.torr_area_m2.sum()/1e6:.1f} km²')

Tørrfall polygons: 214,628, total area 1932.9 km²
CPU times: user 11.3 s, sys: 2.19 s, total: 13.4 s
Wall time: 1min 18s


In [4]:
%%time
pred = gpd.read_parquet(PRED_URL).to_crs('EPSG:25833')
pred['class'] = pred['LM_DK'].map(LM_DK_TO_CLASS)
pred = pred.dropna(subset=['class'])[['class', 'geometry']]
# Restrict to tørrfall bbox so overlay only touches polygons that can intersect.
xmin, ymin, xmax, ymax = torr.total_bounds
pred = pred.cx[xmin:xmax, ymin:ymax].copy()
print(f'Prediction polygons intersecting tørrfall bbox: {len(pred):,}')

Prediction polygons intersecting tørrfall bbox: 439,986
CPU times: user 4.03 s, sys: 1.27 s, total: 5.3 s
Wall time: 22.2 s


In [5]:
%%time
inter = gpd.overlay(torr, pred, how='intersection', keep_geom_type=True)
inter['area_m2'] = inter.geometry.area

CPU times: user 41min 34s, sys: 11.7 s, total: 41min 46s
Wall time: 41min 47s


In [6]:
total = torr['torr_area_m2'].sum() / 1e6
by_cls = inter.groupby('class')['area_m2'].sum().reindex(CLASS_ORDER, fill_value=0) / 1e6
no_pred = total - by_cls.sum()
national = pd.DataFrame({
    'area_km2': list(by_cls.values) + [no_pred, total],
    'pct_of_torrfall': [v / total * 100 for v in by_cls.values] + [no_pred / total * 100, 100.0],
}, index=CLASS_ORDER + ['no_prediction', 'total']).round(2)
national

,area_km2,pct_of_torrfall
soft,448.24,23.19
mixture,572.49,29.62
hard,875.40,45.29
no_prediction,36.82,1.90
total,1932.95,100.00
